# verify_hydrodynamics

**Module:** Hydrodynamic Estimator  
**Equations:** EQ-HYD-001…017  
**References:** Fossen 2nd Ed. (slender body); ITTC-1957; Hoerner cross-flow / streamlined body

In [ ]:
import math
from auv_fin_design.domain.vehicle.model import VehicleModel, MissionModel
from auv_fin_design.domain.hydrodynamics.estimator import estimate_hydrodynamics, ittc_1957_cf
from auv_fin_design.domain.hydrodynamics.yaw_damping import (
    crossflow_yaw_reference,
    yaw_hydrodynamic_moment,
)

v = VehicleModel(length=1.35, diameter=0.1685, mass=24.0, water='freshwater')
m = MissionModel(design_speed=1.5, turning_radius=6.0, turn_establishment_time=4.0)

rho, nu = v.fluid.density, v.fluid.kinematic_viscosity
V, L, D, R = m.speed, v.length, v.diameter, v.radius

Re_L = V * L / nu
q = 0.5 * rho * V**2
Cf = ittc_1957_cf(Re_L)
Y_vdot = rho * math.pi * R**2 * L
N_rdot = rho * math.pi * R**2 * L**3 / 12
r_op = V / m.turning_radius
N_cross = crossflow_yaw_reference(rho, D, L, cd_cross=1.0)
# Fossen polynomial at design (v=0): N = N_rrr * r^3
N_poly = -N_cross / r_op * r_op**3

h = estimate_hydrodynamics(v, m)
N_prod = yaw_hydrodynamic_moment(h.design_lateral_speed_mps, r_op, h.yaw_damping)
pairs = {
    'Re_L': (Re_L, h.re_length),
    'q': (q, h.dynamic_pressure),
    'Cf': (Cf, h.cf_ittc),
    'Y_vdot': (Y_vdot, h.Y_vdot),
    'N_rdot': (N_rdot, h.N_rdot),
    'N_rrr': (-N_cross / r_op, h.yaw_damping.N_rrr),
    'N_poly@r_op': (N_poly, N_prod),
}
for name, (manual, prod) in pairs.items():
    err = abs(manual - prod) / max(abs(manual), 1e-30)
    print(f'{name}: manual={manual:.6g} prod={prod:.6g} err={err:.3e} {"PASS" if err<=0.01 else "FAIL"}')